# Sub-Category: PGA-UNet vs SAM-Med2D

| Section | Nội dung |
|---|---|
| **0** | SAM-Med2D chạy toàn bộ per-polygon → sort Dice → **Easy top-50** · **Medium mid-50** · **Hard bottom-50** |
| **1** | SAM-Med2D chạy 150 samples → 6 metrics × 3 nhóm + show **10 ảnh/nhóm** (30 total) |
| **2** | PGA-UNet chạy 150 samples → 6 metrics × 3 nhóm + show **10 ảnh/nhóm** (30 total) |
| **Agg** | Bảng so sánh SAM-Med2D vs PGA × 3 nhóm + bar chart |

Metrics đều tính tại **256×256** (cả SAM và PGA đều chạy ở 256×256 để so sánh công bằng). Visualization: **Ảnh** | **GT** | **Dự đoán** *(Dice/IoU/CBL)* | **Overlap** *(Pre/Rec/HD95)*

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP (Kaggle / Colab compatible)
# ══════════════════════════════════════════════════════
import os, sys, gdown

# Tự nhận diện môi trường
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

# ── Clone repos ───────────────────────────────────────────────────────
REPO_PGA = 'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'
REPO_SAM = 'https://github.com/OpenGVLab/SAM-Med2D/'

if not os.path.exists(f'{BASE}/pga-repo'):
    !git clone -q -b TN_B_ON {REPO_PGA} {BASE}/pga-repo
print(f'  ✅ pga-repo')

if not os.path.exists(f'{BASE}/SAM-Med2D'):
    !git clone -q {REPO_SAM} {BASE}/SAM-Med2D
print(f'  ✅ SAM-Med2D  (sys.path, không cần pip install -e .)')

# ── Dataset ───────────────────────────────────────────────────────────
DS_PATH = f'{BASE}/pga-repo/dataset_BTXRD'
if not os.path.exists(DS_PATH):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   f'{BASE}/dataset_BTXRD.zip', quiet=False)
    !unzip -oq {BASE}/dataset_BTXRD.zip -d {BASE}/pga-repo/
print('  ✅ Dataset ready')

# ── Checkpoints ────────────────────────────────────────────────────────
os.makedirs(f'{BASE}/pga-repo/checkpoints', exist_ok=True)
for cid, fpath in [
    ('1hGK6_4WCmH4Tzd2XJcerQFYMeDLSiYsg', f'{BASE}/SAM-Med2D/best_sam.pth'),          # finetuned SAMMed2D
    ('1GUgKTTOtAs7thFc7R1TPRTKc0dVSlW_e', f'{BASE}/pga-repo/checkpoints/pga_unet_expB_best.pth'),
]:
    if not os.path.exists(fpath):
        gdown.download(f'https://drive.google.com/uc?id={cid}', fpath, quiet=False)
    print(f'  ✅ {os.path.basename(fpath)}  {os.path.getsize(fpath)//1024}KB')

print('\n✅ Setup hoàn tất')

In [ ]:

# Fix build_sam.py — dùng Python write thay %%writefile để BASE được resolve đúng trên cả Kaggle lẫn Colab
import os as _os
_BASE = '/kaggle/working' if _os.path.exists('/kaggle/working') else '/content'
_build_sam_path = f'{_BASE}/SAM-Med2D/segment_anything/build_sam.py'
_os.makedirs(_os.path.dirname(_build_sam_path), exist_ok=True)

_build_sam_code = '''import torch
from functools import partial
from .modeling import ImageEncoderViT, MaskDecoder, PromptEncoder, Sam, TwoWayTransformer
from torch.nn import functional as F

def build_sam_vit_b(args):
    return _build_sam(
        encoder_embed_dim=768, encoder_depth=12,
        encoder_num_heads=12, encoder_global_attn_indexes=[2,5,8,11],
        image_size=args.image_size, checkpoint=args.sam_checkpoint,
        encoder_adapter=args.encoder_adapter,
    )

build_sam       = build_sam_vit_b
build_sam_vit_h = build_sam_vit_b
build_sam_vit_l = build_sam_vit_b

sam_model_registry = {
    \'default\': build_sam_vit_b,
    \'vit_h\':   build_sam_vit_b,
    \'vit_l\':   build_sam_vit_b,
    \'vit_b\':   build_sam_vit_b,
}

def _build_sam(encoder_embed_dim, encoder_depth, encoder_num_heads,
               encoder_global_attn_indexes, image_size, checkpoint, encoder_adapter):
    prompt_embed_dim = 256; vit_patch_size = 16
    image_embedding_size = image_size // vit_patch_size
    sam = Sam(
        image_encoder=ImageEncoderViT(
            depth=encoder_depth, embed_dim=encoder_embed_dim, img_size=image_size,
            mlp_ratio=4, norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
            num_heads=encoder_num_heads, patch_size=vit_patch_size, qkv_bias=True,
            use_rel_pos=True, global_attn_indexes=encoder_global_attn_indexes,
            window_size=14, out_chans=prompt_embed_dim, adapter_train=encoder_adapter,
        ),
        prompt_encoder=PromptEncoder(
            embed_dim=prompt_embed_dim,
            image_embedding_size=(image_embedding_size, image_embedding_size),
            input_image_size=(image_size, image_size), mask_in_chans=16,
        ),
        mask_decoder=MaskDecoder(
            num_multimask_outputs=3,
            transformer=TwoWayTransformer(
                depth=2, embedding_dim=prompt_embed_dim, mlp_dim=2048, num_heads=8),
            transformer_dim=prompt_embed_dim, iou_head_depth=3, iou_head_hidden_dim=256,
        ),
        pixel_mean=[123.675, 116.28, 103.53],
        pixel_std=[58.395, 57.12, 57.375],
    )
    if checkpoint is not None:
        with open(checkpoint, \'rb\') as f:
            sd = torch.load(f, map_location=\'cpu\', weights_only=False)
        try:
            sam.load_state_dict(sd.get(\'model\', sd), strict=False)
        except Exception:
            sd2 = sam.state_dict()
            new = {k: v for k, v in sd.items() if k in sd2}
            pos = new.get(\'image_encoder.pos_embed\')
            tok = image_size // vit_patch_size
            if pos is not None and pos.shape[1] != tok:
                pos = F.interpolate(pos.permute(0,3,1,2), (tok,tok),
                                    mode=\'bilinear\', align_corners=False)
                new[\'image_encoder.pos_embed\'] = pos.permute(0,2,3,1)
            sd2.update(new); sam.load_state_dict(sd2)
        print(f\'Loaded {checkpoint}\')
    return sam
'''

with open(_build_sam_path, 'w') as f:
    f.write(_build_sam_code)
print(f'✅ Patched {_build_sam_path}')


In [ ]:

# ══════════════════════════════════════════════════════
# CELL 3 — Load models + Shared utilities
# ══════════════════════════════════════════════════════
# Import standard libs TRƯỚC sys.path.insert để tránh torch bị shadow bởi pga-repo/SAM-Med2D
import cv2, json as _json, glob, argparse
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as Rect

# ── Sau đó mới thêm custom paths ─────────────────────
import sys
os.chdir(BASE)
sys.path.insert(0, f'{BASE}/pga-repo')
sys.path.insert(0, f'{BASE}/SAM-Med2D')
for _k in list(sys.modules.keys()):
    if 'segment_anything' in _k:
        del sys.modules[_k]

from models.networks.prompt_unet_2D import PGA_UNet
from segment_anything import sam_model_registry

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
S_PGA, S_SAM, ZOOM_R = 256, 256, 0.30
IMG_DIR  = f'{BASE}/pga-repo/dataset_BTXRD/test/images'
JSON_DIR = f'{BASE}/pga-repo/dataset_BTXRD/test/annotations'
KEYS     = ['dice','iou','precision','recall','hd95','cbl']
SAM_MEAN = np.array([123.675,116.28,103.53],dtype=np.float32)/255.
SAM_STD  = np.array([58.395,57.12,57.375],  dtype=np.float32)/255.

pga=PGA_UNet(in_channels=1,n_classes=1,use_encoder_prompt=True).to(DEVICE)
pga.load_state_dict(torch.load(f'{BASE}/pga-repo/checkpoints/pga_unet_expB_best.pth',
                                map_location=DEVICE,weights_only=True))
pga.eval(); print('✅ PGA loaded')

args_sam=argparse.Namespace(image_size=S_SAM,encoder_adapter=True,
                             sam_checkpoint=f'{BASE}/SAM-Med2D/best_sam.pth')
sam_model=sam_model_registry['vit_b'](args_sam).to(DEVICE)
sam_model.eval(); print('✅ SAM-Med2D loaded')

def extract_lcc(m):
    if m.sum()==0: return m
    n,lbl,st,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8)
    return m if n<=1 else (lbl==(1+np.argmax(st[1:,cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_metrics(prob,gt,eps=1e-6):
    pm=extract_lcc((prob>0.5).astype(np.float32)); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(S_PGA)
    if p.any() and g.any():
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95,cbl=cbl,mask=pm)

def plateau_heatmap(bbox,S=256):
    x1=max(0,int(bbox[0])-5);y1=max(0,int(bbox[1])-5)
    x2=min(S,int(bbox[2])+5);y2=min(S,int(bbox[3])+5)
    hm=np.zeros((S,S),dtype=np.float32)
    if x2>x1 and y2>y1: hm[y1:y2,x1:x2]=1.0; hm=cv2.GaussianBlur(hm,(31,31),0)
    return hm

def infer_sam_512(img_bgr, box_256):
    img256=cv2.resize(img_bgr,(S_SAM,S_SAM))
    gray256=cv2.cvtColor(img256,cv2.COLOR_BGR2GRAY)
    rgb=cv2.cvtColor(gray256,cv2.COLOR_GRAY2RGB).astype(np.float32)/255.
    img_t=torch.from_numpy((rgb-SAM_MEAN)/SAM_STD).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
    box_t=torch.tensor([[list(box_256)]],dtype=torch.float32,device=DEVICE)
    with torch.no_grad():
        emb=sam_model.image_encoder(img_t)
        se,de=sam_model.prompt_encoder(points=None,boxes=box_t,masks=None)
        low,_=sam_model.mask_decoder(image_embeddings=emb,
                                      image_pe=sam_model.prompt_encoder.get_dense_pe(),
                                      sparse_prompt_embeddings=se,
                                      dense_prompt_embeddings=de,multimask_output=False)
    prob256=torch.sigmoid(F.interpolate(low,(S_SAM,S_SAM),mode='bilinear',align_corners=False))[0,0].cpu().numpy()
    return cv2.resize(prob256,(S_PGA,S_PGA),interpolation=cv2.INTER_LINEAR)

def infer_pga_512(img_np_512, bbox_512):
    hm=plateau_heatmap(bbox_512)
    img_t=torch.from_numpy(img_np_512).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
    hm_t=torch.from_numpy(hm).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        return torch.sigmoid(pga(img_t,hm_t))[0,0].cpu().numpy()

def print_metrics(label,mlist):
    m={k:np.mean([r[k] for r in mlist]) for k in KEYS}
    print(f"  {label:<26} Dice={m['dice']:.4f} IoU={m['iou']:.4f} Pre={m['precision']:.4f}"
          f" Rec={m['recall']:.4f} HD95={m['hd95']:.2f} CBL={m['cbl']:.4f}")
    return m

def visualize(records, title, n=10, show_bbox=True):
    recs=records[:n]; nr=len(recs)
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(title,fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Ảnh gốc','Ground Truth','Dự đoán\n(Dice / IoU / CBL)','Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,rec in enumerate(recs):
        gray=np.clip((rec['img_np']*0.5+0.5)*255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=rec['gt_512']; pm=rec['m']['mask']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32); o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        if show_bbox:
            bx=rec['bbox_512']
            ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],lw=1.5,edgecolor='lime',facecolor='none'))
        ax.set_ylabel(f"{rec['stem']}\n#{rec['sidx']}",fontsize=7,rotation=0,labelpad=70,va='center')
        ax.axis('off')
        axes[row,1].imshow(ov(gt,[30,120,255])); axes[row,1].axis('off')
        m=rec['m']
        axes[row,2].imshow(ov(pm,[220,60,60]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
    plt.tight_layout()
    fname='_'.join(title.split('|')[:2]).lower().replace(' ','_')[:30]+'.png'
    plt.savefig(fname,dpi=100,bbox_inches='tight'); plt.show(); print(f'  ✅ {fname}')

# ── Load all per-polygon samples ─────────────────────────────────
all_raw=[]
for jf in sorted(glob.glob(f'{JSON_DIR}/*.json')):
    stem=os.path.splitext(os.path.basename(jf))[0]
    img_path=next((f'{IMG_DIR}/{stem}{e}' for e in ('.png','.jpg','.jpeg')
                   if os.path.exists(f'{IMG_DIR}/{stem}{e}')),None)
    if not img_path: continue
    img_bgr=cv2.imread(img_path); H0,W0=img_bgr.shape[:2]
    data=_json.load(open(jf,encoding='utf-8'))
    for sidx,shp in enumerate([s for s in data.get('shapes',[]) if s.get('shape_type')=='polygon']):
        pts=np.array(shp['points'],dtype=np.float32)
        sx512,sy512=S_PGA/W0,S_PGA/H0
        pts_512=pts*np.array([sx512,sy512])
        gt_512=np.zeros((S_PGA,S_PGA),dtype=np.uint8)
        cv2.fillPoly(gt_512,[pts_512.astype(np.int32)],255)
        xs,ys=pts_512[:,0],pts_512[:,1]
        bw,bh=xs.max()-xs.min(),ys.max()-ys.min()
        b512=[max(0,xs.min()-bw*ZOOM_R),max(0,ys.min()-bh*ZOOM_R),
              min(S_PGA,xs.max()+bw*ZOOM_R),min(S_PGA,ys.max()+bh*ZOOM_R)]
        b256=[v*(S_SAM/S_PGA) for v in b512]
        img512=cv2.resize(img_bgr,(S_PGA,S_PGA))
        gray512=cv2.cvtColor(img512,cv2.COLOR_BGR2GRAY)
        img_np=(gray512.astype(np.float32)/255.-0.5)/0.5
        all_raw.append(dict(stem=stem,sidx=sidx,
                            img_bgr=img_bgr.copy(),
                            img_np=img_np.copy(),
                            gt_512=(gt_512/255.).astype(np.float32),
                            bbox_512=b512,bbox_256=b256))
nho_idx=[]; mo_idx=[]; ro_idx=[]; SEC={}
print(f'✅ {len(all_raw)} per-polygon samples loaded')


def calc_metrics_img(prob, gt, eps=1e-6, IMG_S=512):
    """Image-level: NO LCC (nhiều polygon). GT union + max-merge prob."""
    pm = (prob > 0.5).astype(np.float32)
    gm = (gt   > 0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p, g = pm.astype(bool), gm.astype(bool); hd95 = float(IMG_S)
    if p.any() and g.any():
        from scipy.ndimage import binary_erosion, distance_transform_edt
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl, mask=pm)


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 0 — Phân nhóm 3 nhóm lâm sàng (IMAGE-LEVEL)
# Nhóm theo ảnh gốc (không phải per-polygon):
#   nho_stems  : 50 ảnh có TỔNG diện tích tổn thương nhỏ nhất
#   mo_stems   : 50 ảnh tiếp theo mà SAM-Med2D (image-level) Dice tệ nhất
#   ro_stems   : 50 ảnh còn lại có tổng diện tích lớn nhất
# ══════════════════════════════════════════════════════
from collections import OrderedDict
import numpy as np

# Gom per-polygon raw thành per-image
stem_groups = OrderedDict()
for s in all_raw:
    st = s['stem']
    if st not in stem_groups:
        stem_groups[st] = dict(stem=st, samples=[], img_bgr=s['img_bgr'], img_np=s['img_np'])
    stem_groups[st]['samples'].append(s)

all_stems = list(stem_groups.keys())
N_img = len(all_stems)
print(f'  Tổng: {len(all_raw)} polygons | {N_img} ảnh gốc')

# Tổng diện tích GT mỗi ảnh (union)
def stem_gt_union(grp):
    gt = np.zeros((S_PGA, S_PGA), dtype=np.float32)
    for s in grp['samples']: np.maximum(gt, s['gt_512'], out=gt)
    return gt

total_areas = {st: stem_gt_union(stem_groups[st]).sum() for st in all_stems}
area_rank   = sorted(all_stems, key=lambda s: total_areas[s])
nho_stems   = area_rank[:50]
nho_set     = set(nho_stems)
print(f'  nho (N=50): tổng area {total_areas[area_rank[0]]:.0f} → {total_areas[area_rank[49]]:.0f} px')

# SAM image-level Dice cho tất cả ảnh (merge per-image)
print(f'  Chạy SAM-Med2D image-level trên {N_img} ảnh...')
sam_img_dice = {}
with torch.no_grad():
    for st in tqdm(all_stems, 'Sec0-SAM-img'):
        grp = stem_groups[st]
        gt_union = stem_gt_union(grp)
        prob_max = np.zeros((S_PGA, S_PGA), dtype=np.float32)
        for s in grp['samples']:
            prob512 = infer_sam_512(s['img_bgr'], s['bbox_256'])
            np.maximum(prob_max, prob512, out=prob_max)
        eps = 1e-6; pm=(prob_max>0.5).astype(float); gm=(gt_union>0.5).astype(float)
        tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
        sam_img_dice[st] = float((2*tp+eps)/(2*tp+fp+fn+eps))

remaining_1   = [s for s in all_stems if s not in nho_set]
mo_stems      = sorted(remaining_1, key=lambda s: sam_img_dice[s])[:50]
mo_set        = set(mo_stems)
remaining_2   = [s for s in all_stems if s not in nho_set and s not in mo_set]
ro_stems      = sorted(remaining_2, key=lambda s: -total_areas[s])[:50]

print(f'  mo (N=50):  SAM Dice {sam_img_dice[mo_stems[0]]:.4f} → {sam_img_dice[mo_stems[-1]]:.4f}')
print(f'  ro (N=50):  tổng area {total_areas[ro_stems[0]]:.0f} → {total_areas[ro_stems[-1]]:.0f} px')
SEC = {}


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 1 — SAM-Med2D  (IMAGE-LEVEL)
# Mỗi ảnh: chạy SAM trên từng polygon bbox → max-merge → metric ảnh
# ══════════════════════════════════════════════════════
from collections import OrderedDict

res_sam = {'nho':{},'mo':{},'ro':{}}
with torch.no_grad():
    for key,lbl,stems in [('nho','Tổn thương nhỏ',nho_stems),
                           ('mo', 'Biên giới mờ',  mo_stems),
                           ('ro', 'Rõ nét',        ro_stems)]:
        for st in tqdm(stems, f'SAM-{lbl}'):
            grp = stem_groups[st]
            gt_union = np.zeros((S_PGA,S_PGA),dtype=np.float32)
            prob_max = np.zeros((S_PGA,S_PGA),dtype=np.float32)
            prompts  = []
            for s in grp['samples']:
                np.maximum(gt_union, s['gt_512'], out=gt_union)
                p512 = infer_sam_512(s['img_bgr'], s['bbox_256'])
                np.maximum(prob_max, p512, out=prob_max)
            res_sam[key][st] = dict(stem=st, img_np=grp['img_np'],
                                    gt=gt_union, prob=prob_max, prompts=[],
                                    n_samples=len(grp['samples']),
                                    m=calc_metrics_img(prob_max, gt_union))

bar='='*74; print(f'\\n{bar}\\n  SECTION 1 — SAM-Med2D (image-level)\\n{bar}')
SEC['sam']={}
for key,lbl in [('nho','Tổn thương nhỏ'),('mo','Biên giới mờ'),('ro','Tổn thương rõ nét')]:
    SEC['sam'][key] = print_metrics(f'{lbl} (N=50)', [v['m'] for v in res_sam[key].values()])

def visualize_img(records, title, n=10):
    """Image-level vis: merged pred + GT union."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from IPython.display import display as _disp
    recs = records[:n]; nr = len(recs)
    if nr == 0: return
    fig, axes = plt.subplots(nr, 5, figsize=(20, 4*nr), squeeze=False)
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.002)
    for j,t in enumerate(['Ảnh gốc','Prompts (merged)','Dự đoán (merged)','GT (union)','TP/FP/FN']):
        axes[0,j].set_title(t, fontsize=9, fontweight='bold')
    for row, rec in enumerate(recs):
        img_np = np.clip((rec['img_np']*0.5+0.5), 0, 1)
        gt     = (rec['gt']>0.5).astype(float)
        pred   = (rec['prob']>0.5).astype(float)
        pm_merged = np.max(np.stack(rec['prompts'],0),0) if rec.get('prompts') else np.zeros_like(img_np)
        tp=(pred*gt).sum(); fp=(pred*(1-gt)).sum(); fn=((1-pred)*gt).sum(); e=1e-6
        dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp+e)/(tp+fp+fn+e))
        pre=float((tp+e)/(tp+fp+e));         rec_=float((tp+e)/(tp+fn+e))
        n_poly = rec.get('n_samples', '?')
        bg = np.stack([img_np]*3,-1)
        axes[row,0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        axes[row,0].set_ylabel(f"{rec['stem']} [{n_poly}p]\nDice={dice:.3f}", fontsize=7)
        axes[row,1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        axes[row,1].imshow(pm_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
        pr_ov=bg.copy(); pr_ov[...,0]=np.clip(pr_ov[...,0]+pred*0.55,0,1); pr_ov[...,1]=np.clip(pr_ov[...,1]-pred*0.2,0,1)
        axes[row,2].imshow(pr_ov); axes[row,2].set_title(f'Dice={dice:.3f} IoU={iou:.3f}',fontsize=7,color='darkred',pad=2)
        gt_ov=bg.copy(); gt_ov[...,1]=np.clip(gt_ov[...,1]+gt*0.55,0,1); gt_ov[...,0]=np.clip(gt_ov[...,0]-gt*0.2,0,1)
        axes[row,3].imshow(gt_ov)
        inter=bg.copy()*0.25
        inter[...,1]=np.clip(inter[...,1]+pred*gt*0.9,0,1)
        inter[...,0]=np.clip(inter[...,0]+pred*(1-gt)*1.0,0,1)
        inter[...,2]=np.clip(inter[...,2]+(1-pred)*gt*1.0,0,1)
        axes[row,4].imshow(inter); axes[row,4].set_title(f'Pre={pre:.3f} Rec={rec_:.3f}',fontsize=7,color='saddlebrown',pad=2)
        for ax in axes[row]: ax.axis('off')
    fig.legend(handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')],
               loc='lower center',ncol=3,fontsize=8,bbox_to_anchor=(0.5,-0.004))
    plt.tight_layout(); _disp(fig); plt.close(fig)

for key,lbl,stems in [('nho','Tổn thương nhỏ',nho_stems),('mo','Biên giới mờ',mo_stems),('ro','Tổn thương rõ nét',ro_stems)]:
    show_s = stems[-10:][::-1] if key=='mo' else stems[:10]
    visualize_img([res_sam[key][s] for s in show_s if s in res_sam[key]],
                  f'SAM-Med2D | {lbl} — 10 ảnh (image-level)', n=10)


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 2 — PGA-UNet  (IMAGE-LEVEL)
# Mỗi ảnh: chạy PGA trên từng polygon → max-merge → metric ảnh
# ══════════════════════════════════════════════════════
res_pga = {'nho':{},'mo':{},'ro':{}}
with torch.no_grad():
    for key,lbl,stems in [('nho','Tổn thương nhỏ',nho_stems),
                           ('mo', 'Biên giới mờ',  mo_stems),
                           ('ro', 'Tổn thương rõ nét',ro_stems)]:
        for st in tqdm(stems, f'PGA-{lbl}'):
            grp = stem_groups[st]
            gt_union = np.zeros((S_PGA,S_PGA),dtype=np.float32)
            prob_max = np.zeros((S_PGA,S_PGA),dtype=np.float32)
            prompts  = []
            for s in grp['samples']:
                np.maximum(gt_union, s['gt_512'], out=gt_union)
                p512 = infer_pga_512(s['img_np'], s['bbox_512'])
                hm   = plateau_heatmap(s['bbox_512'])
                np.maximum(prob_max, p512, out=prob_max)
                prompts.append(hm)
            res_pga[key][st] = dict(stem=st, img_np=grp['img_np'],
                                    gt=gt_union, prob=prob_max, prompts=prompts,
                                    n_samples=len(grp['samples']),
                                    m=calc_metrics_img(prob_max, gt_union))

bar='='*74; print(f'\\n{bar}\\n  SECTION 2 — PGA-UNet (image-level)\\n{bar}')
SEC['pga']={}
for key,lbl in [('nho','Tổn thương nhỏ'),('mo','Biên giới mờ'),('ro','Tổn thương rõ nét')]:
    SEC['pga'][key] = print_metrics(f'{lbl} (N=50)', [v['m'] for v in res_pga[key].values()])
for key,lbl,stems in [('nho','Tổn thương nhỏ',nho_stems),('mo','Biên giới mờ',mo_stems),('ro','Tổn thương rõ nét',ro_stems)]:
    show_s = stems[-10:][::-1] if key=='mo' else stems[:10]
    visualize_img([res_pga[key][s] for s in show_s if s in res_pga[key]],
                  f'PGA-UNet | {lbl} — 10 ảnh (image-level)', n=10)


In [ ]:
# ══════════════════════════════════════════════════════
# CELL TỔNG HỢP — So sánh SAM vs PGA × 3 nhóm
# ══════════════════════════════════════════════════════
import csv, os

bar='═'*80
print(f'\n{bar}')
print('  BẢNG SO SÁNH  |  Tổn thương nhỏ · Biên giới mờ · Tổn thương rõ nét')
print(f'{bar}')
print(f"  {'Nhóm':<20} {'Model':<16} {'Dice':>7} {'IoU':>7} {'Pre':>7} {'Rec':>7} {'HD95':>8} {'CBL':>7}")
print(f"  {'─'*85}")

csv_rows=[]
for key,lbl in [('nho','Tổn thương nhỏ'),('mo','Biên giới mờ'),('ro','Tổn thương rõ nét')]:
    for model,mkey in [('SAM-Med2D','sam'),('PGA-UNet','pga')]:
        m = SEC[mkey][key]
        print(f"  {lbl:<20} {model:<16}"
              f" {m['dice']:>7.4f} {m['iou']:>7.4f} {m['precision']:>7.4f}"
              f" {m['recall']:>7.4f} {m['hd95']:>8.2f} {m['cbl']:>7.4f}")
        csv_rows.append([lbl, model, 50] + [f"{m[k]:.4f}" for k in KEYS])
    dp=SEC['pga'][key]['dice']; ds=SEC['sam'][key]['dice']
    print(f"  {'':20} {'Δ PGA−SAM':<16} {dp-ds:>+7.4f}")
    print(f"  {'─'*85}")
print(bar)

os.makedirs('results', exist_ok=True)
with open('results/subcat_pga_vs_sam.csv','w',newline='',encoding='utf-8') as f:
    csv.writer(f).writerows([['group','model','N']+KEYS]+csv_rows)
print('→ CSV: results/subcat_pga_vs_sam.csv')

# Bar chart
x_labels=['Tổn thương nhỏ','Biên giới mờ','Tổn thương rõ nét']
x=np.arange(3); w=0.35
fig,ax=plt.subplots(figsize=(10,5))
for i,(label,mkey,clr) in enumerate([('SAM-Med2D','sam','#ffb74d'),('PGA-UNet','pga','#ef5350')]):
    vals=[SEC[mkey][k]['dice'] for k in ['nho','mo','ro']]
    bars=ax.bar(x+(i-0.5)*w,vals,w,label=label,color=clr,edgecolor='white')
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,v+0.012,f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(x_labels,fontsize=10)
ax.set_ylabel('Dice'); ax.set_ylim(0,1.12)
ax.legend(fontsize=11); ax.grid(axis='y',alpha=0.3)
ax.set_title('PGA-UNet vs SAM-Med2D — Phân tích theo đặc tính tổn thương',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('results/subcat_sam_bar.png',dpi=130,bbox_inches='tight'); plt.show()
print('→ Plot: results/subcat_sam_bar.png')